In [1]:
from frameworks.LightenDiffusion.models.LightenDiffusion import Stage1
from frameworks.LightenDiffusion.models.decom import ImageEncoder, ImageDecoder, RetinexDecomposition
from dataset_registery.registery import DatasetManager
from torchsummary import summary
from frameworks.LightenDiffusion.train_lightendiffusion import Stage1Trainer
from eda.helpers.data_helpers import split_dataloader
from eda.helpers.training_helpers import get_optimizer
from frameworks.LightenDiffusion.losses import ctdn_loss_wrapper
import matplotlib.pyplot as plt
import torch
import os
%load_ext autoreload
%autoreload 2

/home/grads/o/omarkhater/projects/lle-generative-priors/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_name = "SICE_paired"
dataset_id = "okhater/SICE"
source = "huggingface"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device used : {device}")

Device used : cuda


In [3]:
registry = DatasetManager()
registry.initialize_dataset(
    name=data_name,
    dataset_id=dataset_id,
    splits=["train", "test"],
    dataset_type="paired",
    hf_cache_dir=f"../../datasets/{data_name}",
)

train_loader = registry.get_dataloader(data_name, "train", batch_size=16, shuffle=True)
test_loader = registry.get_dataloader(data_name, "test", batch_size=16, shuffle=True)
train_loader, val_loader = split_dataloader(train_loader, split_ratio=0.2)

loaders = {
    "train": train_loader, 
    "val": val_loader, 
    "test": test_loader
    }
for name, loader in loaders.items():
    total_samples = len(loader.dataset)
    print(f"Loader: {name}, Total samples: {total_samples}")
    for low_imgs, label in loader:
        print(f"Batch contains {len(low_imgs)} low image samples.")
        print("Low images batch shape:", low_imgs.shape)
        print("Label batch shape:", label.shape)
        break
    print("==="*20)

Loader: train, Total samples: 207
Batch contains 16 low image samples.
Low images batch shape: torch.Size([16, 2, 3, 256, 256])
Label batch shape: torch.Size([16, 3, 256, 256])
Loader: val, Total samples: 51
Batch contains 16 low image samples.
Low images batch shape: torch.Size([16, 2, 3, 256, 256])
Label batch shape: torch.Size([16, 3, 256, 256])
Loader: test, Total samples: 46
Batch contains 16 low image samples.
Low images batch shape: torch.Size([16, 2, 3, 256, 256])
Label batch shape: torch.Size([16, 3, 256, 256])


In [4]:
stage1_model = Stage1(
    encoder=ImageEncoder(64),
    decoder=ImageDecoder(64),
    decomposer=RetinexDecomposition(),
)
stage1_model.to(device)
summary(stage1_model, (2 , 3, 256, 256), batch_size=1)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 256, 256]           4,864
            Conv2d-2          [1, 64, 256, 256]         102,464
            Conv2d-3          [1, 64, 256, 256]          36,928
         LeakyReLU-4          [1, 64, 256, 256]               0
            Conv2d-5          [1, 64, 256, 256]          36,928
            Conv2d-6          [1, 64, 256, 256]           4,160
         Res_block-7          [1, 64, 256, 256]               0
            Conv2d-8          [1, 64, 128, 128]          36,928
            Conv2d-9         [1, 128, 128, 128]          73,856
        LeakyReLU-10         [1, 128, 128, 128]               0
           Conv2d-11         [1, 128, 128, 128]         147,584
           Conv2d-12         [1, 128, 128, 128]           8,320
        Res_block-13         [1, 128, 128, 128]               0
           Conv2d-14           [1, 128,

In [5]:
stage1_optimizer = get_optimizer(stage1_model)

In [6]:
trainer1 = Stage1Trainer(
    model=stage1_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=ctdn_loss_wrapper,
    optimizer=stage1_optimizer,
    device=device,
    val_frequency =  1,
    patience = 2,
    num_epochs=5
)
best_stage1, metrics1 = trainer1.train()

Epoch 1/5: train=0.1918, val=0.1905


Epoch 2/5: train=0.1878, val=0.1734


Epoch 3/5: train=0.1864, val=0.1785


Epoch 4/5: train=0.1911, val=0.2152
Early stopping


In [7]:
directory = "/home/grads/o/omarkhater/projects/lle-generative-priors/frameworks/LightenDiffusion/trained_models/stage1/"
file_path = os.path.join(directory, "best_stage1.pth")
if not os.path.exists(directory):
    os.makedirs(directory)
torch.save(best_stage1.state_dict(), file_path)